# 🧹 Lab 3: Data Cleaning & Preprocessing
### AI/ML Internship | Day 3

**Student:** Arslan Hashmi  
**Institution:** National University of Technology (NUTECH)  
**Track:** AI/ML Internship Program  

---

Real-world data is almost never ready to use straight away — it usually has missing values, duplicate rows, inconsistent formatting, wrong data types, and outliers. This lab walks through identifying and fixing each of these problems using **Pandas** and **NumPy**.

**After this lab you will be able to:**
- Identify common data quality issues in a real dataset
- Handle missing values using multiple strategies
- Remove duplicate records
- Standardize inconsistent text and date formats
- Fix incorrect data types
- Detect and handle outliers using the IQR method

> 💡 *In most real projects, data cleaning and preprocessing take up more time than model building itself.*

## Why Data Cleaning Matters

Before any model can learn from data, the data itself needs to be trustworthy. Messy data leads directly to wrong conclusions and poor model performance — a model cannot tell the difference between a genuine value and a data-entry mistake unless you clean it first.

The most common problems you will run into are:

| Issue | Description | Example |
|-------|-------------|--------|
| **Missing values** | Empty cells (`NaN`) from incomplete data entry | Age left blank |
| **Duplicate records** | Same row appears more than once | Identical employee entries |
| **Inconsistent formatting** | Same category written differently | `"HR"` vs `"hr "` vs `"Hr"` |
| **Incorrect data types** | Numbers stored as text or wrong signs | Phone numbers stored as negative |
| **Outliers** | Values far outside expected range | Age of 200 or salary of $1 |

We will fix each of these, one at a time, using the messy HR dataset below.

## 📦 Setup & Loading the Dataset

We work with **Messy_Employee_dataset.csv** — a deliberately imperfect employee records file containing exactly the kinds of issues described above.

Columns include: Employee ID, names, age, department-region, employment status, join date, salary, email, phone, performance score, and remote-work flag.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load the messy dataset
df = pd.read_csv("data/Messy_Employee_dataset.csv")

print("✅ Dataset loaded successfully!")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head(10)

## Step 1 — Inspecting the Data

Before fixing anything, always inspect the dataset first: how many rows and columns it has, what data types each column has, and how many missing values exist.

In [ ]:
print("Dataset shape:", df.shape)
print("\n" + "="*60)
df.info()

**Exercise:** Print the number of missing values in each column, and separately, the number of fully duplicated rows in the dataset.

In [ ]:
missing_counts = None
duplicate_count = None

### YOUR CODE HERE ###
missing_counts = df.isnull().sum()
duplicate_count = df.duplicated().sum()
### END ###

print("Missing values per column:")
print(missing_counts)
print("\nDuplicate rows:", duplicate_count)

**Observations from inspection:**
- **Age** has 211 missing values (~20.7% of records)
- **Salary** has 24 missing values
- **Phone** numbers are stored as *negative* integers (data entry error)
- No exact duplicate rows exist
- `Department_Region` combines department + location (e.g., `DevOps-California`)
- `Performance_Score` is categorical (Poor / Average / Good / Excellent)

## Step 2 — Removing Duplicates

Exact duplicate rows add no new information and can skew any summary statistics or model trained on the data.

**Exercise:** Create `df_no_dupes`, a copy of `df` with all fully duplicated rows removed. Keep the first occurrence of each.

In [ ]:
df_no_dupes = None

### YOUR CODE HERE ###
df_no_dupes = df.drop_duplicates(keep="first").copy()
### END ###

print("Before:", df.shape[0], "rows | After:", df_no_dupes.shape[0], "rows")
print(f"Removed {df.shape[0] - df_no_dupes.shape[0]} duplicate row(s).")

## Step 3 — Handling Missing Values

There is no single correct way to handle missing values — the right strategy depends on the column and the situation:

- **Drop rows** when very few rows have missing values and losing them won't hurt the analysis.
- **Fill with the mean/median** for numeric columns, when a reasonable estimate is better than losing the row entirely.
- **Fill with a placeholder** (e.g., `"Unknown"`) for categorical/text columns, when the missing information is itself meaningful.

**Exercise:** On `df_no_dupes`, fill missing `Age` values with the column's median, fill missing `Salary` values with the column's mean, and confirm no remaining missing values in critical columns. Store the result in `df_clean`. 

In [ ]:
df_clean = df_no_dupes.copy()

### YOUR CODE HERE ###
# Fill Age with median (robust to outliers)
median_age = df_clean["Age"].median()
df_clean["Age"] = df_clean["Age"].fillna(median_age)

# Fill Salary with mean
mean_salary = df_clean["Salary"].mean()
df_clean["Salary"] = df_clean["Salary"].fillna(mean_salary)
### END ###

print("Missing values after cleaning:")
print(df_clean.isnull().sum())
print(f"\nAge filled with median = {median_age}")
print(f"Salary filled with mean  = {mean_salary:.2f}")

## Step 4 — Standardizing Inconsistent Text

Look closely at name and department columns — the same value can appear with different capitalization or extra whitespace. To a computer, these are different strings unless you standardize them.

**Exercise:** On `df_clean`:
1. Strip extra whitespace and convert `First_Name` / `Last_Name` to title case
2. Create a clean `Full_Name` column
3. Split `Department_Region` into separate `Department` and `Region` columns and standardize them
4. Fix known acronym casing (DevOps, HR)

In [ ]:
### YOUR CODE HERE ###
# Clean names
df_clean["First_Name"] = df_clean["First_Name"].str.strip().str.title()
df_clean["Last_Name"]  = df_clean["Last_Name"].str.strip().str.title()
df_clean["Full_Name"]  = df_clean["First_Name"] + " " + df_clean["Last_Name"]

# Standardize the combined department-region field
df_clean["Department_Region"] = df_clean["Department_Region"].str.strip().str.title()

# Split into Department and Region
df_clean[["Department", "Region"]] = (
    df_clean["Department_Region"]
    .str.split("-", n=1, expand=True)
)
df_clean["Department"] = df_clean["Department"].str.strip()
df_clean["Region"]     = df_clean["Region"].str.strip().str.title()

# Restore proper acronyms after .title()
dept_fix = {
    "Devops": "DevOps",
    "Hr": "HR",
    "Cloud Tech": "Cloud Tech"
}
df_clean["Department"] = df_clean["Department"].replace(dept_fix)
### END ###

print("Unique Departments:", sorted(df_clean["Department"].unique()))
print("Unique Regions:   ", sorted(df_clean["Region"].unique()))
print("\nSample of cleaned names & departments:")
df_clean[["Full_Name", "Department", "Region"]].head(8)

**Expected Department values:**
```
['Admin', 'Cloud Tech', 'DevOps', 'Finance', 'HR', 'Sales']
```

## Step 5 — Fixing Incorrect Data Types

Several columns need type correction:
- **Phone** values are stored as *negative* integers (clear data-entry bug) → convert to positive strings
- **Join_Date** is text with mixed formats → parse to proper `datetime`
- **Performance_Score** is categorical text → also create a numeric mapping for aggregation

**Exercise:** Implement the cleaning function / transformations below.

In [ ]:
def clean_phone(value):
    """Convert negative phone numbers to positive string representation."""
    ### YOUR CODE HERE ###
    if pd.isna(value):
        return np.nan
    return str(abs(int(value)))
    ### END ###

df_clean["Phone"] = df_clean["Phone"].apply(clean_phone)

# Parse dates (handles M/D/YYYY and similar formats gracefully)
df_clean["Join_Date"] = pd.to_datetime(df_clean["Join_Date"], errors="coerce", format="mixed")

# Map performance to numeric scores for later aggregation
perf_map = {
    "Poor": 1,
    "Average": 2,
    "Good": 3,
    "Excellent": 4
}
df_clean["Performance_Numeric"] = df_clean["Performance_Score"].map(perf_map)

print("Phone dtype     :", df_clean["Phone"].dtype)
print("Join_Date dtype :", df_clean["Join_Date"].dtype)
print("NaT (unparsed)  :", df_clean["Join_Date"].isna().sum())
print("\nSample cleaned phones & dates:")
df_clean[["Employee_ID", "Phone", "Join_Date", "Performance_Score", "Performance_Numeric"]].head(8)

## Step 6 — Detecting Outliers (IQR Method)

An outlier is a value far outside the range you would reasonably expect. One common way to detect outliers is the **IQR (Interquartile Range)** method:

$$
\text{lower bound} = Q1 - 1.5 \times IQR \\
\text{upper bound} = Q3 + 1.5 \times IQR
$$

Values outside these bounds are flagged.

**Exercise:** Using the IQR method on the `Age` column, identify which rows are outliers and store them in `age_outliers`. Then replace outlier ages with the column's median.

In [ ]:
Q1 = df_clean["Age"].quantile(0.25)
Q3 = df_clean["Age"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

age_outliers = None

### YOUR CODE HERE ###
age_outliers = df_clean[
    (df_clean["Age"] < lower_bound) | (df_clean["Age"] > upper_bound)
].copy()
### END ###

print(f"Age IQR bounds: [{lower_bound:.1f}, {upper_bound:.1f}]")
print(f"Number of age outliers: {len(age_outliers)}")
if len(age_outliers) > 0:
    print(age_outliers[["Employee_ID", "Full_Name", "Age"]])
else:
    print("✅ No age outliers detected (ages are tightly clustered: 25–40).")

In [ ]:
# Replace any age outliers with the median (safe even if zero outliers)
median_age = df_clean["Age"].median()
df_clean.loc[
    (df_clean["Age"] < lower_bound) | (df_clean["Age"] > upper_bound),
    "Age"
] = median_age

print("Age summary after outlier treatment:")
print(df_clean["Age"].describe().round(2))

**What to remember from this lab:**
- Always inspect a dataset (`.info()`, `.isnull().sum()`, `.duplicated().sum()`) before doing anything else
- There is no single "correct" way to handle missing values — the right strategy depends on context
- Inconsistent text formatting must be standardized before grouping or filtering by it
- Data types must be correct before any numeric operation will work
- The IQR method is one simple, common way to flag outliers

---
## 📝 Lab Tasks

Complete the following tasks using the fully cleaned `df_clean` dataframe.

### Task 1 — Full Pipeline Recap

In this lab I performed a complete end-to-end cleaning pipeline on the messy employee dataset. First I inspected the data and discovered missing ages/salaries, negative phone numbers, and a combined department-region field. I removed any exact duplicate rows (none were present), then imputed missing ages with the median and missing salaries with the mean. Next I standardized all name fields to title case, created a clean Full_Name, and split + normalized the Department_Region column into proper Department and Region values while restoring acronyms such as DevOps and HR. I corrected data types by turning phone numbers into positive strings and parsing join dates into true datetime objects, and mapped performance scores to a numeric scale. Finally I applied the IQR method to detect age outliers (none existed in this dataset) and prepared the data for analysis. The result is a trustworthy, analysis-ready dataframe.

### Task 2 — Salary Outliers

Repeat the IQR outlier-detection method from Step 6, but this time apply it to the cleaned `Salary` column. Print any rows flagged as outliers.

In [ ]:
# Task 2
Q1_sal = df_clean["Salary"].quantile(0.25)
Q3_sal = df_clean["Salary"].quantile(0.75)
IQR_sal = Q3_sal - Q1_sal
lower_sal = Q1_sal - 1.5 * IQR_sal
upper_sal = Q3_sal + 1.5 * IQR_sal

salary_outliers = df_clean[
    (df_clean["Salary"] < lower_sal) | (df_clean["Salary"] > upper_sal)
].copy()

print(f"Salary IQR bounds: [${lower_sal:,.2f}, ${upper_sal:,.2f}]")
print(f"Number of salary outliers: {len(salary_outliers)}")

if len(salary_outliers) > 0:
    print("\nFlagged salary outliers:")
    display(salary_outliers[["Employee_ID", "Full_Name", "Department", "Salary"]].sort_values("Salary"))
else:
    print("✅ No salary outliers detected — all values fall comfortably within the 1.5×IQR range.")

### Task 3 — Department Summary

Using `df_clean`, compute the average `Salary` and average `Performance_Numeric` per `Department`, in a single `.groupby().agg()` call.

In [ ]:
# Task 3
dept_summary = (
    df_clean
    .groupby("Department")
    .agg(
        avg_salary=("Salary", "mean"),
        avg_performance=("Performance_Numeric", "mean"),
        employee_count=("Employee_ID", "count")
    )
    .round(2)
    .sort_values("avg_salary", ascending=False)
)

print("📊 Average Salary & Performance by Department")
print("=" * 55)
dept_summary

### Task 4 — Export the Cleaned Data

Save `df_clean` to a new file named `employee_records_clean.csv` using `.to_csv()`, without the index column.

In [ ]:
# Task 4
# Select a clean, analysis-ready column order
export_cols = [
    "Employee_ID", "Full_Name", "First_Name", "Last_Name", "Age",
    "Department", "Region", "Status", "Join_Date", "Salary",
    "Email", "Phone", "Performance_Score", "Performance_Numeric",
    "Remote_Work"
]

df_clean[export_cols].to_csv("data/employee_records_clean.csv", index=False)
print("✅ Cleaned dataset exported to: data/employee_records_clean.csv")
print(f"   Rows: {df_clean.shape[0]} | Columns: {len(export_cols)}")

### Task 5 — Reflection

The most challenging step for me was **standardizing the Department_Region column**. The original field mixed two pieces of information (department + geographic region) with inconsistent separators and casing. After splitting, the `.str.title()` method turned useful acronyms such as "DevOps" into "Devops" and "HR" into "Hr", so I had to add an explicit mapping dictionary to restore the correct professional casing. This taught me that automated string methods are powerful but still require domain-aware post-processing when dealing with real organizational data.

---
## 🎯 Final Clean Dataset Snapshot

In [ ]:
print("Final cleaned dataset overview")
print("=" * 50)
print(f"Shape          : {df_clean.shape}")
print(f"Missing values : {df_clean.isnull().sum().sum()}")
print(f"Duplicate rows : {df_clean.duplicated().sum()}")
print("\nColumn dtypes:")
print(df_clean[export_cols].dtypes)
print("\nFirst 5 cleaned rows:")
df_clean[export_cols].head()

---

**Lab completed by:** Arslan Hashmi  
**Date:** July 2026  
**Tools:** Python 3 · Pandas · NumPy · Google Colab / Jupyter  

> *"Clean data is the foundation of every trustworthy insight."*